# 03 — Float Types: float32, float16, int8, binary

Como reduzir o uso de memória de embeddings em até **32x** com impacto mínimo na qualidade?

| Tipo | Bits | Memória | Precisão relativa | Use Case |
|------|------|---------|-------------------|----------|
| float32 | 32 | 100% | 100% (baseline) | Desenvolvimento, qualidade máxima |
| float16 | 16 | 50% | ~99% | GPU inference, produção com GPU |
| int8 | 8 | 25% | ~97% | CPU produção, Qdrant scalar quantization |
| binary | 1 | 3% | ~60-80% | Large-scale approximate search |

---
**O que vamos explorar:**
1. Como cada tipo armazena números
2. Cálculo real de memória
3. Impacto na similaridade cosine
4. Demo com módulo `src/embeddings/quantization.py`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import struct
import sys
sys.path.insert(0, '..')

from sentence_transformers import SentenceTransformer
from src.embeddings.quantization import (
    quantize_vectors, dequantize_vectors,
    memory_comparison_table, cosine_similarity_preservation
)

print('Modulos carregados!')

## 3.1 Como cada tipo representa números

In [ ]:
valor = 0.123456789

f32 = np.float32(valor)
f16 = np.float16(valor)
i8_approx = np.int8(np.clip(round(valor * 127), -128, 127))

print(f'Valor original: {valor}')
print(f'float32: {f32}  (erro: {abs(float(f32) - valor):.2e})  [4 bytes]')
print(f'float16: {f16}  (erro: {abs(float(f16) - valor):.2e})  [2 bytes]')
print(f'int8:    {i8_approx} -> deq: {float(i8_approx)/127:.6f}  (erro: {abs(float(i8_approx)/127 - valor):.2e})  [1 byte]')

bits = ''.join(f'{b:08b}' for b in struct.pack('>f', valor))
print(f'\nfloat32 em bits: {bits}')
print(f'  Sign: {bits[0]} | Exponent: {bits[1:9]} ({int(bits[1:9], 2) - 127}) | Mantissa: {bits[9:]}')

## 3.2 Cálculo real de memória com embeddings reais

In [ ]:
model = SentenceTransformer('all-mpnet-base-v2')  # 768d

docs = [
    'Embeddings sao representacoes vetoriais de texto.',
    'Redes neurais aprendem padroes complexos nos dados.',
    'O Qdrant e um banco de dados vetorial open-source.',
    'RAG combina retrieval com geracao de linguagem.',
    'Quantizacao reduz memoria com minima perda de qualidade.',
] * 200  # 1000 documentos

embeddings = model.encode(docs, normalize_embeddings=True, show_progress_bar=True)
print(f'\nEmbeddings: {embeddings.shape} | dtype: {embeddings.dtype}')

In [ ]:
table = memory_comparison_table(embeddings)

rows = []
bytes_per = {'float32': 4, 'float16': 2, 'int8': 1, 'binary': 0.125}
for dtype, stats in table.items():
    rows.append({
        'Tipo': dtype,
        'Bytes/elemento': bytes_per[dtype],
        '1K docs (768d) MB': f"{stats['mb']:.2f}",
        'Projecao 1M docs GB': f"{stats['gb'] * 1000:.2f}",
        'Reducao vs float32': 'baseline' if dtype == 'float32' else f"{stats['reduction_pct']:.0f}%",
        'Fator': '1x' if dtype == 'float32' else f"{stats['reduction_ratio']:.0f}x",
    })

df = pd.DataFrame(rows)
print('Uso de memoria para 1000 documentos a 768 dimensoes:\n')
print(df.to_string(index=False))

In [ ]:
# Visualizacao
tipos = list(table.keys())
memorias_mb = [table[t]['mb'] for t in tipos]
cores = ['#3498db', '#2ecc71', '#f39c12', '#e74c3c']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(tipos, memorias_mb, color=cores, edgecolor='white', linewidth=2)
for bar, mem in zip(bars, memorias_mb):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{mem:.2f}MB', ha='center', va='bottom', fontweight='bold')

ax.set_title('Memoria por Tipo de Float (1000 docs x 768 dimensoes)', fontsize=13)
ax.set_ylabel('Memoria (MB)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 3.3 Impacto na Qualidade: Preservacao de Similaridade Cosine

In [ ]:
print('Medindo preservacao de similaridade cosine...\n')

quality_results = []
for dtype in ['float32', 'float16', 'int8', 'binary']:
    result = cosine_similarity_preservation(embeddings, dtype, sample_size=500)
    quality_results.append(result)
    print(f"{dtype:8s}: erro medio={result['mean_abs_error']:.5f} | "
          f"preservacao={result['preservation_pct']:.2f}% | "
          f"correlacao={result['correlation']:.5f}")

print('\nInterpretacao:')
print('  float16: quase identico ao float32 (erro ~0.001)')
print('  int8: pequena degradacao (~1-3%), otimo para producao')
print('  binary: degradacao significativa — so para busca aproximada em larga escala')

In [ ]:
# Scatter: similaridade original vs quantizada
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, (dtype, color) in enumerate([('float16', '#2ecc71'), ('int8', '#f39c12'), ('binary', '#e74c3c')]):
    n_sample = 200
    idx = np.random.choice(len(embeddings), n_sample, replace=False)
    sample = embeddings[idx]

    orig_norm = sample / np.linalg.norm(sample, axis=1, keepdims=True)
    orig_sims = (orig_norm @ orig_norm.T).flatten()

    quant = quantize_vectors(sample, dtype)
    dequant = dequantize_vectors(quant, dtype)
    q_norm = dequant / (np.linalg.norm(dequant, axis=1, keepdims=True) + 1e-9)
    q_sims = (q_norm @ q_norm.T).flatten()

    axes[i].scatter(orig_sims[::5], q_sims[::5], alpha=0.2, s=2, color=color)
    axes[i].plot([-1, 1], [-1, 1], 'k--', alpha=0.5, linewidth=1)

    corr = np.corrcoef(orig_sims, q_sims)[0, 1]
    mae = np.mean(np.abs(orig_sims - q_sims))

    axes[i].set_title(f'{dtype}\ncorr={corr:.4f}, MAE={mae:.4f}', fontsize=11)
    axes[i].set_xlabel('Similaridade Original (float32)')
    axes[i].set_ylabel('Similaridade Quantizada')
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Preservacao de Similaridade Cosine apos Quantizacao', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3.4 Quantizacao no Qdrant (Scalar Quantization)

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    ScalarQuantization, ScalarQuantizationConfig, ScalarType
)

client = QdrantClient(host='localhost', port=6333)

# Collection com int8 quantization nativa do Qdrant
if client.collection_exists('demo_quantization'):
    client.delete_collection('demo_quantization')
client.create_collection(
    collection_name='demo_quantization',
    vectors_config=VectorParams(size=768, distance=Distance.COSINE),
    quantization_config=ScalarQuantization(
        scalar=ScalarQuantizationConfig(
            type=ScalarType.INT8,
            quantile=0.99,
            always_ram=True,
        )
    )
)

points = [
    PointStruct(id=i, vector=embeddings[i].tolist(), payload={'text': docs[i]})
    for i in range(50)
]
client.upsert(collection_name='demo_quantization', points=points)

print('Collection criada com int8 quantizacao!')
print('Vantagens:')
print('  - Vetores armazenados como int8 (4x menos memoria)')
print('  - Busca ~2x mais rapida')
print('  - Recall tipico: 97-99%')
print('  - API identica — transparente para o usuario')

## Resumo: Guia de Decisao de Float Type

```
Precisa reduzir memoria?
├── Nao → float32 (qualidade maxima)
├── ~50% reducao → float16 (GPU disponivel? Excelente)
├── ~75% reducao → int8 / Scalar Quantization (RECOMENDADO para CPU)
└── >96% reducao → binary (so large-scale, aceitar 60-80% recall)
```

**Recomendacao para RAG de producao:**
> Use **int8 Scalar Quantization no Qdrant** — 75% menos memoria, 97%+ recall, 2x velocidade.

## Proximos passos
- [04 — Distance Metrics](04_distance_metrics.html)
- [02 — Qdrant Quantization](../02_vector_databases/03_quantization.html)